# 68 - Held-out PRO160 Q-Planning evaluation, worker 3/4

Shard 3 of 4 over the reserved 160 position-perturbation LIBERO-PRO identities. Every identity runs three matched arms: stock PI0.5, frozen Q10 planning, and frozen Q50 planning. Stock uses 10 Euler steps; planners use 64 candidates, 3 Euler steps, top 16 Q-softmax averaging. Every arm executes 10 actions then replans. No online learning or historical rollouts are used. Videos and frames are off.

This worker has 40 identities and 120 rollouts and prints exact matched SR every 10 completed identities. Set EPISODE_LIMIT = 1 only for an optional smoke test.

## 1. Setup

In [ ]:
EXTRAS = 'sim'
SETUP_ENV = True
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

## 2. Checkpoints and fixed shard

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
SHARD_COUNT = 4
SHARD_INDEX = 3
EPISODE_LIMIT = None
CANDIDATE_BATCH_SIZE = 8
OUTPUT_ROOT = Path('/content/drive/MyDrive/pnp_qplanning_corrector')
Q10_CHECKPOINT_PATH = None
Q50_CHECKPOINT_PATH = None

def resolve_checkpoint(horizon, explicit):
    if explicit is not None:
        path = Path(explicit)
        if not path.is_file(): raise FileNotFoundError(path)
        return path
    matches = sorted(OUTPUT_ROOT.glob(f'pcpcds-*/q{horizon}_full/checkpoint_step_008000.pt'))
    if len(matches) != 1:
        raise ValueError(f'Expected exactly one Q{horizon} full checkpoint; found {len(matches)}: {[str(p) for p in matches]}. Set Q{horizon}_CHECKPOINT_PATH explicitly.')
    return matches[0]

Q10_CHECKPOINT_PATH = resolve_checkpoint(10, Q10_CHECKPOINT_PATH)
Q50_CHECKPOINT_PATH = resolve_checkpoint(50, Q50_CHECKPOINT_PATH)
print({'shard': f'{SHARD_INDEX}/{SHARD_COUNT}', 'episode_limit': EPISODE_LIMIT,
       'q10_checkpoint': str(Q10_CHECKPOINT_PATH), 'q50_checkpoint': str(Q50_CHECKPOINT_PATH)})

## 3. Run the three matched arms

In [ ]:
from pnp.qplanning_eval_experiment import run_qplanning_heldout_worker
report = run_qplanning_heldout_worker(
    q10_checkpoint_path=Q10_CHECKPOINT_PATH, q50_checkpoint_path=Q50_CHECKPOINT_PATH,
    shard_count=SHARD_COUNT, shard_index=SHARD_INDEX, episode_limit=EPISODE_LIMIT,
    candidate_batch_size=CANDIDATE_BATCH_SIZE)
report

## 4. Persisted three-arm audit

In [ ]:
from pnp.qplanning_eval_experiment import validate_qplanning_heldout_sentinel
validate_qplanning_heldout_sentinel(
    q10_checkpoint_id=report['q10_checkpoint_id'], q50_checkpoint_id=report['q50_checkpoint_id'],
    experiment=report['experiment'])